# CONSUMO DE ENERGIA CPE6

## CARGA DE BASES

In [26]:
from utils import UTILS
from data_processing import PREPROP_MERGER
import pandas as pd
ut=UTILS()
prepm=PREPROP_MERGER()

# Leer una hoja por nombre (recomendado motor calamine)
df_produccion = pd.read_excel(
    "./in/produccion_cpe6_new.xlsx",
    sheet_name="Sheet1",
)

# Leer una hoja por nombre (recomendado motor calamine)
df_consumo = pd.read_excel(
    "./in/consumo_energetico_CPE6.xlsx",
    sheet_name="Sheet1",
)

# Leer una hoja por nombre (recomendado motor calamine)
df_fo = pd.read_excel(
    "./in/df_consumoDiarioFO_CPE6_global.xlsx",
    sheet_name="Sheet1",
)
# Leer una hoja por nombre (recomendado motor calamine)
df_fo_capmax = pd.read_excel(
    "./in/df_CapMax_CPE6_global.xlsx",
    sheet_name="Sheet1",
)


In [27]:
df_produccion['CLUSTER'].unique()

array(['CPE6-1X', 'CPE6H-2', 'CPE6H-5', 'CPE6H-3', 'HAMACA-2', 'HAMACA-8',
       'HAMACA-7', 'AMANECER-1', 'COPLERO-1', 'HAMACA-29', 'HAMACA-30',
       'HAMACA-64', 'HAMACA-65', 'HAMACA NORTE-1', 'HAMACA-73',
       ' MANACACIAS-2'], dtype=object)

In [28]:
df_produccion.columns

Index(['BLOQUE', 'CAMPO', 'POZO', 'CLUSTER', 'Fecha', 'BOPD', 'BWPD', 'BFPD'], dtype='object')

## PROCESOS

In [29]:
# PRODUCCIÓN
df_produccion_llave = df_produccion.copy()

# Normaliza fecha (como ya venías haciendo)
df_produccion_llave["Fecha"] = (
    pd.to_datetime(df_produccion_llave["Fecha"], dayfirst=True, errors="coerce")
      .dt.tz_localize(None).dt.normalize()
)

# Llave canónica
df_produccion_llave["llave_clusters"] = df_produccion_llave["CLUSTER"].map(prepm.canon_llave_clusters)


C:\Users\User C360\AppData\Local\Temp\ipykernel_17720\601540828.py:6: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(df_produccion_llave["Fecha"], dayfirst=True, errors="coerce")


In [30]:
# ENERGÍA
df_energia_llave = df_consumo.copy()
# Llave canónica desde la ubicación (usar la original o la normalizada indistinto)
df_energia_llave["llave_clusters"] = df_energia_llave["UBICACIÓN"].map(prepm.canon_llave_clusters)

In [31]:
# CONSUMO FO
df_fo_llave = df_fo.copy()

# Llave canónica desde 'cluster'
df_fo_llave["llave_clusters"] = df_fo_llave["CLUSTER"].map(prepm.canon_llave_clusters)

df_fo_llave=prepm.merge_into(df_fo_llave,df_fo_capmax,['CLUSTER'],['cluster'])
df_fo_llave["#Tannques/dia"]=1

df_fo_llave.to_excel('./out/EDA_consumoFO.xlsx', index=False)
df_energia_llave.to_excel('./out/EDA_consumoEnergy.xlsx', index=False)

In [32]:
df_fo_llave.columns

Index(['CLUSTER', 'FECHA', 'stock_ayer', 'stock_hoy', 'consumo',
       'autonomia_hrs', 'autonomia_dias', 'consumo_calculado', 'month', 'year',
       'month_year', 'ones', 'llave_clusters', 'capacidad', '#Tannques/dia'],
      dtype='object')

In [33]:
df_energia_llave_sums=ut.totalizar_mixta(df_energia_llave, keys=["fecha","llave_clusters"])
df_fo_llave_sums=ut.totalizar_mixta(df_fo_llave, keys=["FECHA","llave_clusters"])

In [34]:
cols_energy = ['#dias_op',"CORRIENTE [A]", "ENERGIA", "FACTOR DE CARGA [%]", "POTENCIA [Kw]","POTENCIA NOMINAL [KW]",
               "PRESIÓN ACEITE (PSI)", "TEMPERATURA MOTOR (F°)"]

df_energia_llave_avg = ut.group_mean_with_suffix(
    df_energia_llave,
    index_cols=["fecha", "llave_clusters"],
    numeric_cols=cols_energy,
    round_decimals=3
)

cols_fo = ["#Tannques/dia","capacidad", "consumo", "consumo_calculado", "stock_ayer","stock_hoy"]
df_fo_llave_avg = ut.group_mean_with_suffix(
    df_fo_llave,
    index_cols=["FECHA", "llave_clusters"],
    numeric_cols=cols_fo,
    round_decimals=3
)


c:\Users\User C360\Documents\Optimizacion_EnergeticaCPE6\utils.py:964: FutureWarning: The provided callable <function mean at 0x000001FD67E81C60> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  X.groupby(list(index_cols), dropna=False, sort=sort_keys)
c:\Users\User C360\Documents\Optimizacion_EnergeticaCPE6\utils.py:964: FutureWarning: The provided callable <function mean at 0x000001FD67E81C60> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  X.groupby(list(index_cols), dropna=False, sort=sort_keys)
c:\Users\User C360\Documents\Optimizacion_EnergeticaCPE6\utils.py:964: FutureWarning: The provided callable <function mean at 0x000001FD67E81C60> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will b

In [35]:
df_energia_produccion=prepm.merge_into(df_produccion_llave,df_energia_llave_sums,['Fecha','llave_clusters'],['fecha','llave_clusters'])
df_energia_produccion_fo=prepm.merge_into(df_energia_produccion,df_fo_llave_sums,['Fecha','llave_clusters'],['FECHA','llave_clusters'])


In [36]:
df_energia_produccion_fo=prepm.merge_into(df_energia_produccion_fo,df_energia_llave_avg,['Fecha','llave_clusters'],['fecha','llave_clusters'])
df_energia_produccion_fo=prepm.merge_into(df_energia_produccion_fo,df_fo_llave_avg,['Fecha','llave_clusters'],['FECHA','llave_clusters'])


In [37]:
df_energia_produccion_fo.to_excel('./out/EDA_produccion_energia.xlsx', index=False)